In [1]:
import pandas as pd

## Load data

In [2]:
df_meta_train = pd.read_csv("../../data/monte_outputs/pancancer/monte_pancancer_meta_with_predictions_train.csv")
df_meta_test = pd.read_csv("../../data/monte_outputs/pancancer/monte_pancancer_meta_with_predictions_test.csv")
trained_metric = "CPE"

## Calculate the correlation between different metrics and purity estimations

In [3]:
def calculate_correlations_by_cancer(df, trained_metric):
    pred_metric = f"predicted_{trained_metric}"
    metrics = ["ABSOLUTE", "ESTIMATE", "LUMP", "CPE"]
    cancer_types = df_meta_train["Cancer.type"].unique()

    result = pd.DataFrame(index=cancer_types, columns=metrics + ["n_samples"], dtype=float)

    for cancer_type in cancer_types:
        df_cancer = df[df["Cancer.type"] == cancer_type]
        result.loc[cancer_type, "n_samples"] = len(df_cancer)

        for metric in metrics:
            # use only valid paired values for correlation
            pair = df_cancer[[pred_metric, metric]].dropna()

            if len(pair) < 2:
                corr = float("nan")
            elif pair[pred_metric].std() == 0 or pair[metric].std() == 0:
                # No variance in one or both columns
                corr = float("nan")
            else:
                corr = pair[pred_metric].corr(pair[metric])

            result.loc[cancer_type, metric] = corr

    return result

In [4]:
df_corr_train = calculate_correlations_by_cancer(df_meta_train, trained_metric)
df_corr_test = calculate_correlations_by_cancer(df_meta_test, trained_metric)

## Save results

In [5]:
df_corr_train.to_csv("../../data/monte_outputs/pancancer/monte_pancancer_metrics_correlation_results_train.csv", index=True)
df_corr_test.to_csv("../../data/monte_outputs/pancancer/monte_pancancer_metrics_correlation_results_test.csv", index=True)